## Sector Focused Training

We will perform the daily training based on the sector data only, to monitor the time series.

In [2]:
import pandas as pd
import pandas_ta as ta

In [3]:
fin_sec_data=pd.read_csv('feature_engineering/stocks_sectors/financial_services/financial_services_sector')

In [4]:
fin_sec_data

,date,sector,weighted_sector_returns,n_stocks_contributing,sector_index,sector_volume
0,2017-01-03,financial_services,0.000000,20.0,100.000000,48125518.0
1,2017-01-04,financial_services,0.353432,20.0,100.353432,26705191.0
2,2017-01-05,financial_services,-0.182223,20.0,100.170565,66991076.0
3,2017-01-06,financial_services,1.397196,20.0,101.570144,97521276.0
4,2017-01-09,financial_services,3.645896,20.0,105.273286,99013315.0
...,...,...,...,...,...,...
2258,2026-07-13,financial_services,-0.746327,35.0,2695.019313,290562049.0
2259,2026-07-14,financial_services,0.906947,35.0,2719.461701,163509442.0
2260,2026-07-15,financial_services,0.779799,35.0,2740.668037,258362267.0
2261,2026-07-16,financial_services,1.505893,35.0,2781.939574,261040578.0


We have to shift the volume data forward to avoid data leakage.

In [5]:
fin_sec_data['sector_volume']=fin_sec_data['sector_volume'].shift(1)

In [6]:
fin_sec_data

,date,sector,weighted_sector_returns,n_stocks_contributing,sector_index,sector_volume
0,2017-01-03,financial_services,0.000000,20.0,100.000000,NaN
1,2017-01-04,financial_services,0.353432,20.0,100.353432,48125518.0
2,2017-01-05,financial_services,-0.182223,20.0,100.170565,26705191.0
3,2017-01-06,financial_services,1.397196,20.0,101.570144,66991076.0
4,2017-01-09,financial_services,3.645896,20.0,105.273286,97521276.0
...,...,...,...,...,...,...
2258,2026-07-13,financial_services,-0.746327,35.0,2695.019313,242350071.0
2259,2026-07-14,financial_services,0.906947,35.0,2719.461701,290562049.0
2260,2026-07-15,financial_services,0.779799,35.0,2740.668037,163509442.0
2261,2026-07-16,financial_services,1.505893,35.0,2781.939574,258362267.0


In [7]:
fin_sec_data=fin_sec_data.dropna()

In [8]:
fin_sec_feat=fin_sec_data[['weighted_sector_returns','n_stocks_contributing','sector_index','sector_volume']]

In [9]:
import pandas_ta as ta
## Functions for feature engineering
def lag_returns(data: pd.DataFrame,lag_items: list[int] | None = None) -> pd.DataFrame:
    if lag_items is None:
        lag_items=[1]
    for lag in lag_items:
        data[f'sector_lag_return{lag}']=data['weighted_sector_returns'].shift(lag)
    return data
    
def moving_averages(data: pd.DataFrame,averaging_items: list[int] | None = None) -> pd.DataFrame:
    if averaging_items is None:
        averaging_items=[5]
    for avg in averaging_items:
        data[f'sector_index_MA_{avg}']=data['sector_index'].rolling(avg).mean()
    return data

def rolling_volatility(data: pd.DataFrame,volatility_items: list[int] | None = None) -> pd.DataFrame:
    if volatility_items is None:
        volatility_items=[20]
    for vol in volatility_items:
        data[f'sector_returns_volatility_{vol}']=data['weighted_sector_returns'].rolling(vol).std()
    return data

def relative_strength_index(data:pd.DataFrame, rsi_length:int | None=None) -> pd.DataFrame:
     if rsi_length is None:
        rsi_length=14

     data['sector_index_RSI']=ta.rsi(data['sector_index'], length=rsi_length)
     return data

def ma_convergence_divergence(data:pd.DataFrame) -> pd.DataFrame:
     data.ta.macd(close='sector_index', append=True)
     data.rename(columns={'MACD_12_26_9':'sector_MACD','MACDh_12_26_9':'sector_MACD_hist','MACDs_12_26_9':'sector_MACD_signal'},inplace=True)
     return data

def volume_moving_average(data: pd.DataFrame,volume_items: list[int] | None = None) -> pd.DataFrame:
    if volume_items is None:
        volume_items=[20]
    for vol in volume_items:
        data[f'sector_stock_volume_MA_{vol}']=data['sector_volume'].rolling(vol).mean()
    return data

In [10]:
## Apply lagging on the returns

fin_sec_lag=lag_returns(data=fin_sec_feat,lag_items=[1,2,5,10])

In [11]:
fin_sec_lag

,weighted_sector_returns,n_stocks_contributing,sector_index,sector_volume,sector_lag_return1,sector_lag_return2,sector_lag_return5,sector_lag_return10
1,0.353432,20.0,100.353432,48125518.0,NaN,NaN,NaN,NaN
2,-0.182223,20.0,100.170565,26705191.0,0.353432,NaN,NaN,NaN
3,1.397196,20.0,101.570144,66991076.0,-0.182223,0.353432,NaN,NaN
4,3.645896,20.0,105.273286,97521276.0,1.397196,-0.182223,NaN,NaN
5,-1.454119,20.0,103.742486,99013315.0,3.645896,1.397196,NaN,NaN
...,...,...,...,...,...,...,...,...
2258,-0.746327,35.0,2695.019313,242350071.0,-0.556971,0.125740,2.204600,-1.123746
2259,0.906947,35.0,2719.461701,290562049.0,-0.746327,-0.556971,-0.344068,-1.000082
2260,0.779799,35.0,2740.668037,163509442.0,0.906947,-0.746327,1.008166,-1.697222
2261,1.505893,35.0,2781.939574,258362267.0,0.779799,0.906947,0.125740,-1.250020


In [12]:
## Apply moving averages
fin_sec_MA=moving_averages(data=fin_sec_lag,averaging_items=[5,20,50])

In [13]:
fin_sec_MA

,weighted_sector_returns,n_stocks_contributing,sector_index,sector_volume,sector_lag_return1,sector_lag_return2,sector_lag_return5,sector_lag_return10,sector_index_MA_5,sector_index_MA_20,sector_index_MA_50
1,0.353432,20.0,100.353432,48125518.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.182223,20.0,100.170565,26705191.0,0.353432,NaN,NaN,NaN,NaN,NaN,NaN
3,1.397196,20.0,101.570144,66991076.0,-0.182223,0.353432,NaN,NaN,NaN,NaN,NaN
4,3.645896,20.0,105.273286,97521276.0,1.397196,-0.182223,NaN,NaN,NaN,NaN,NaN
5,-1.454119,20.0,103.742486,99013315.0,3.645896,1.397196,NaN,NaN,102.221982,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2258,-0.746327,35.0,2695.019313,242350071.0,-0.556971,0.125740,2.204600,-1.123746,2713.540670,2709.356926,2841.198885
2259,0.906947,35.0,2719.461701,290562049.0,-0.746327,-0.556971,-0.344068,-1.000082,2717.464145,2704.856745,2839.243187
2260,0.779799,35.0,2740.668037,163509442.0,0.906947,-0.746327,1.008166,-1.697222,2720.185104,2702.351004,2836.920167
2261,1.505893,35.0,2781.939574,258362267.0,0.779799,0.906947,0.125740,-1.250020,2730.474568,2704.380561,2834.980562


In [14]:
fin_sec_volatility=rolling_volatility(data=fin_sec_MA,volatility_items=[20,50])

In [15]:
fin_sec_volatility

,weighted_sector_returns,n_stocks_contributing,sector_index,sector_volume,sector_lag_return1,sector_lag_return2,sector_lag_return5,sector_lag_return10,sector_index_MA_5,sector_index_MA_20,sector_index_MA_50,sector_returns_volatility_20,sector_returns_volatility_50
1,0.353432,20.0,100.353432,48125518.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.182223,20.0,100.170565,26705191.0,0.353432,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.397196,20.0,101.570144,66991076.0,-0.182223,0.353432,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3.645896,20.0,105.273286,97521276.0,1.397196,-0.182223,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,-1.454119,20.0,103.742486,99013315.0,3.645896,1.397196,NaN,NaN,102.221982,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2258,-0.746327,35.0,2695.019313,242350071.0,-0.556971,0.125740,2.204600,-1.123746,2713.540670,2709.356926,2841.198885,1.608787,1.378562
2259,0.906947,35.0,2719.461701,290562049.0,-0.746327,-0.556971,-0.344068,-1.000082,2717.464145,2704.856745,2839.243187,1.560957,1.357674
2260,0.779799,35.0,2740.668037,163509442.0,0.906947,-0.746327,1.008166,-1.697222,2720.185104,2702.351004,2836.920167,1.569335,1.346734
2261,1.505893,35.0,2781.939574,258362267.0,0.779799,0.906947,0.125740,-1.250020,2730.474568,2704.380561,2834.980562,1.554344,1.360057


In [16]:
fin_sec_rsi=relative_strength_index(data=fin_sec_volatility,rsi_length=14)

In [17]:
fin_sec_rsi

,weighted_sector_returns,n_stocks_contributing,sector_index,sector_volume,sector_lag_return1,sector_lag_return2,sector_lag_return5,sector_lag_return10,sector_index_MA_5,sector_index_MA_20,sector_index_MA_50,sector_returns_volatility_20,sector_returns_volatility_50,sector_index_RSI
1,0.353432,20.0,100.353432,48125518.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.182223,20.0,100.170565,26705191.0,0.353432,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
3,1.397196,20.0,101.570144,66991076.0,-0.182223,0.353432,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.056744
4,3.645896,20.0,105.273286,97521276.0,1.397196,-0.182223,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.384159
5,-1.454119,20.0,103.742486,99013315.0,3.645896,1.397196,NaN,NaN,102.221982,NaN,NaN,NaN,NaN,56.472274
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2258,-0.746327,35.0,2695.019313,242350071.0,-0.556971,0.125740,2.204600,-1.123746,2713.540670,2709.356926,2841.198885,1.608787,1.378562,43.509761
2259,0.906947,35.0,2719.461701,290562049.0,-0.746327,-0.556971,-0.344068,-1.000082,2717.464145,2704.856745,2839.243187,1.560957,1.357674,46.905288
2260,0.779799,35.0,2740.668037,163509442.0,0.906947,-0.746327,1.008166,-1.697222,2720.185104,2702.351004,2836.920167,1.569335,1.346734,49.728616
2261,1.505893,35.0,2781.939574,258362267.0,0.779799,0.906947,0.125740,-1.250020,2730.474568,2704.380561,2834.980562,1.554344,1.360057,54.769545


In [18]:
## moving average converges divergence
fin_sec_macd=ma_convergence_divergence(data=fin_sec_rsi)

In [19]:
fin_sec_macd

,weighted_sector_returns,n_stocks_contributing,sector_index,sector_volume,sector_lag_return1,sector_lag_return2,sector_lag_return5,sector_lag_return10,sector_index_MA_5,sector_index_MA_20,sector_index_MA_50,sector_returns_volatility_20,sector_returns_volatility_50,sector_index_RSI,sector_MACD,sector_MACD_hist,sector_MACD_signal
1,0.353432,20.0,100.353432,48125518.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.182223,20.0,100.170565,26705191.0,0.353432,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
3,1.397196,20.0,101.570144,66991076.0,-0.182223,0.353432,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.056744,NaN,NaN,NaN
4,3.645896,20.0,105.273286,97521276.0,1.397196,-0.182223,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.384159,NaN,NaN,NaN
5,-1.454119,20.0,103.742486,99013315.0,3.645896,1.397196,NaN,NaN,102.221982,NaN,NaN,NaN,NaN,56.472274,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2258,-0.746327,35.0,2695.019313,242350071.0,-0.556971,0.125740,2.204600,-1.123746,2713.540670,2709.356926,2841.198885,1.608787,1.378562,43.509761,-36.121172,7.827889,-43.949061
2259,0.906947,35.0,2719.461701,290562049.0,-0.746327,-0.556971,-0.344068,-1.000082,2717.464145,2704.856745,2839.243187,1.560957,1.357674,46.905288,-32.419942,9.223295,-41.643237
2260,0.779799,35.0,2740.668037,163509442.0,0.906947,-0.746327,1.008166,-1.697222,2720.185104,2702.351004,2836.920167,1.569335,1.346734,49.728616,-27.458986,11.347401,-38.806387
2261,1.505893,35.0,2781.939574,258362267.0,0.779799,0.906947,0.125740,-1.250020,2730.474568,2704.380561,2834.980562,1.554344,1.360057,54.769545,-19.966959,15.071543,-35.038501


In [20]:
## Stock volume rolling average
fin_sec_vol_MA=volume_moving_average(data=fin_sec_macd,volume_items=[20])

In [21]:
fin_sec_vol_MA

,weighted_sector_returns,n_stocks_contributing,sector_index,sector_volume,sector_lag_return1,sector_lag_return2,sector_lag_return5,sector_lag_return10,sector_index_MA_5,sector_index_MA_20,sector_index_MA_50,sector_returns_volatility_20,sector_returns_volatility_50,sector_index_RSI,sector_MACD,sector_MACD_hist,sector_MACD_signal,sector_stock_volume_MA_20
1,0.353432,20.0,100.353432,48125518.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.182223,20.0,100.170565,26705191.0,0.353432,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
3,1.397196,20.0,101.570144,66991076.0,-0.182223,0.353432,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.056744,NaN,NaN,NaN,NaN
4,3.645896,20.0,105.273286,97521276.0,1.397196,-0.182223,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.384159,NaN,NaN,NaN,NaN
5,-1.454119,20.0,103.742486,99013315.0,3.645896,1.397196,NaN,NaN,102.221982,NaN,NaN,NaN,NaN,56.472274,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2258,-0.746327,35.0,2695.019313,242350071.0,-0.556971,0.125740,2.204600,-1.123746,2713.540670,2709.356926,2841.198885,1.608787,1.378562,43.509761,-36.121172,7.827889,-43.949061,3.401123e+08
2259,0.906947,35.0,2719.461701,290562049.0,-0.746327,-0.556971,-0.344068,-1.000082,2717.464145,2704.856745,2839.243187,1.560957,1.357674,46.905288,-32.419942,9.223295,-41.643237,3.350912e+08
2260,0.779799,35.0,2740.668037,163509442.0,0.906947,-0.746327,1.008166,-1.697222,2720.185104,2702.351004,2836.920167,1.569335,1.346734,49.728616,-27.458986,11.347401,-38.806387,3.267409e+08
2261,1.505893,35.0,2781.939574,258362267.0,0.779799,0.906947,0.125740,-1.250020,2730.474568,2704.380561,2834.980562,1.554344,1.360057,54.769545,-19.966959,15.071543,-35.038501,3.168715e+08


In [22]:
## final draft
fin_sec_final=fin_sec_vol_MA.dropna()

In [23]:
fin_sec_final

,weighted_sector_returns,n_stocks_contributing,sector_index,sector_volume,sector_lag_return1,sector_lag_return2,sector_lag_return5,sector_lag_return10,sector_index_MA_5,sector_index_MA_20,sector_index_MA_50,sector_returns_volatility_20,sector_returns_volatility_50,sector_index_RSI,sector_MACD,sector_MACD_hist,sector_MACD_signal,sector_stock_volume_MA_20
50,1.263442,21.0,105.523744,29332452.0,-2.818687,-0.093507,0.385307,0.649545,106.144992,106.362717,106.833129,0.959811,1.044206,45.207491,-0.257530,-0.147567,-0.109963,1.048182e+08
51,1.291063,21.0,106.886122,57990928.0,1.263442,-2.818687,-0.406173,-1.915182,106.235320,106.367943,106.963783,1.005844,1.056955,51.462200,-0.185331,-0.060294,-0.125037,1.057057e+08
52,1.485459,21.0,108.473871,53225293.0,1.291063,1.263442,0.841357,-0.220672,106.464099,106.473388,107.129849,1.052931,1.073039,57.544758,0.000005,0.100034,-0.100029,1.045779e+08
53,-0.684406,21.0,107.731470,86616367.0,1.485459,1.291063,-0.093507,0.751944,106.564471,106.514549,107.253076,1.062474,1.064605,54.129069,0.085989,0.148814,-0.062825,1.045701e+08
54,-0.630296,21.0,107.052442,101995650.0,-0.684406,1.485459,-2.818687,0.314921,107.133530,106.529197,107.288659,1.072553,0.940365,51.139153,0.098207,0.128826,-0.030619,1.012914e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2258,-0.746327,35.0,2695.019313,242350071.0,-0.556971,0.125740,2.204600,-1.123746,2713.540670,2709.356926,2841.198885,1.608787,1.378562,43.509761,-36.121172,7.827889,-43.949061,3.401123e+08
2259,0.906947,35.0,2719.461701,290562049.0,-0.746327,-0.556971,-0.344068,-1.000082,2717.464145,2704.856745,2839.243187,1.560957,1.357674,46.905288,-32.419942,9.223295,-41.643237,3.350912e+08
2260,0.779799,35.0,2740.668037,163509442.0,0.906947,-0.746327,1.008166,-1.697222,2720.185104,2702.351004,2836.920167,1.569335,1.346734,49.728616,-27.458986,11.347401,-38.806387,3.267409e+08
2261,1.505893,35.0,2781.939574,258362267.0,0.779799,0.906947,0.125740,-1.250020,2730.474568,2704.380561,2834.980562,1.554344,1.360057,54.769545,-19.966959,15.071543,-35.038501,3.168715e+08


#### Training

In [24]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
import numpy as np
## Apply standard scaler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression


In [25]:
## Independent features
X=fin_sec_final.drop(columns=['weighted_sector_returns'])
## dependent features
y=fin_sec_final['weighted_sector_returns']

In [26]:
models_to_train=[LinearRegression,Ridge,Lasso,ElasticNet,XGBRegressor,RandomForestRegressor,AdaBoostRegressor,DecisionTreeRegressor]

In [27]:
# X: independent features, y: sector returns
# CRITICAL: X and y must already be sorted by date, ascending
# X = X.sort_values('date')  # or however df is indexed. Our df is 
# y = y.loc[X.index]

for model in models_to_train:

    print(f'{model} training begins')
    print('')
    tscv = TimeSeriesSplit(n_splits=5)  # 5 sequential folds

    rmse_scores = []
    r2scores_test = []
    r2scores_train=[]
    baseline_rmse = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx] ## This area uses the index to pick the data from X and y
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        ## getting bsaeline
        pred_base = np.full_like(y_test, y_train.mean(), dtype=float)
        baseline_rmse.append(np.sqrt(mean_squared_error(y_test, pred_base)))

        ## Apply standardscaler to standardize the data
        # Fit to the training data (calculates mean and std)
        scaler=StandardScaler()
        scaler.fit(X_train)

        # Transform the training and testing data
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model_obj = model()
        model_obj.fit(X_train_scaled, y_train)
        preds_test = model_obj.predict(X_test_scaled)
        preds_train = model_obj.predict(X_train_scaled)

        rmse = np.sqrt(mean_squared_error(y_test, preds_test))
        rmse_scores.append(rmse)

        r2_test=r2_score(y_true=y_test,y_pred=preds_test)
        r2_train=r2_score(y_true=y_train,y_pred=preds_train)
        r2scores_test.append(r2_test)
        r2scores_train.append(r2_train)

        print(f"Fold {fold}: train={len(X_train)} rows ({X_train.index.min()} to {X_train.index.max()}), "
            f"test={len(X_test)} rows ({X_test.index.min()} to {X_test.index.max()}), RMSE={rmse:.4f}, r2score_test={r2_test:.4f}, r2score_train={r2_train:.4f}")
        
    print(f"\nMean RMSE across folds: {np.mean(rmse_scores):.4f}")
    print(f"\nMean R2_score_test across folds: {np.mean(r2scores_test):.4f}")
    print(f"\nMean R2_score_train across folds: {np.mean(r2scores_train):.4f}")
    print('baseline information: ', baseline_rmse, np.mean(baseline_rmse))
    print('==============================================================================')
    print('==============================================================================')
    print('')

<class 'sklearn.linear_model._base.LinearRegression'> training begins

Fold 0: train=373 rows (50 to 422), test=368 rows (423 to 790), RMSE=0.7469, r2score_test=0.8532, r2score_train=0.8745
Fold 1: train=741 rows (50 to 790), test=368 rows (791 to 1158), RMSE=0.7755, r2score_test=0.7284, r2score_train=0.9040
Fold 2: train=1109 rows (50 to 1158), test=368 rows (1159 to 1526), RMSE=0.8668, r2score_test=0.3386, r2score_train=0.8751
Fold 3: train=1477 rows (50 to 1526), test=368 rows (1527 to 1894), RMSE=5.4822, r2score_test=-6.0596, r2score_train=0.8331
Fold 4: train=1845 rows (50 to 1894), test=368 rows (1895 to 2262), RMSE=3.6320, r2score_test=-4.9389, r2score_train=0.5858

Mean RMSE across folds: 2.3007

Mean R2_score_test across folds: -1.8157

Mean R2_score_train across folds: 0.8145
baseline information:  [np.float64(1.9635693710632107), np.float64(1.4920970933859388), np.float64(1.0658530292384354), np.float64(2.0717283668641078), np.float64(1.4935503653418278)] 1.6173596451787042


I know its not advisable to make the stock_index as the target(check ClaudeAI). But lets give it a test.

In [28]:
## Independent features
X=fin_sec_final.drop(columns=['weighted_sector_returns','sector_index'])
## dependent features
y=fin_sec_final['sector_index']

# X: independent features, y: sector returns
# CRITICAL: X and y must already be sorted by date, ascending
# X = X.sort_values('date')  # or however df is indexed. Our df is 
# y = y.loc[X.index]

for model in models_to_train:

    print(f'{model} training begins')
    print('')
    tscv = TimeSeriesSplit(n_splits=5)  # 5 sequential folds

    rmse_scores = []
    r2scores_test = []
    r2scores_train=[]
    baseline_rmse = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx] ## This area uses the index to pick the data from X and y
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        ## getting bsaeline
        pred_base = np.full_like(y_test, y_train.mean(), dtype=float)
        baseline_rmse.append(np.sqrt(mean_squared_error(y_test, pred_base)))

        ## Apply standardscaler to standardize the data
        # Fit to the training data (calculates mean and std)
        scaler=StandardScaler()
        scaler.fit(X_train)

        # Transform the training and testing data
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model_obj = model()
        model_obj.fit(X_train_scaled, y_train)
        preds_test = model_obj.predict(X_test_scaled)
        preds_train = model_obj.predict(X_train_scaled)

        rmse = np.sqrt(mean_squared_error(y_test, preds_test))
        rmse_scores.append(rmse)

        r2_test=r2_score(y_true=y_test,y_pred=preds_test)
        r2_train=r2_score(y_true=y_train,y_pred=preds_train)
        r2scores_test.append(r2_test)
        r2scores_train.append(r2_train)

        print(f"Fold {fold}: train={len(X_train)} rows ({X_train.index.min()} to {X_train.index.max()}), "
            f"test={len(X_test)} rows ({X_test.index.min()} to {X_test.index.max()}), RMSE={rmse:.4f}, r2score_test={r2_test:.4f}, r2score_train={r2_train:.4f}")
        
    print(f"\nMean RMSE across folds: {np.mean(rmse_scores):.4f}")
    print(f"\nMean R2_score_test across folds: {np.mean(r2scores_test):.4f}")
    print(f"\nMean R2_score_train across folds: {np.mean(r2scores_train):.4f}")
    print('baseline information: ', baseline_rmse, np.mean(baseline_rmse))
    print('==============================================================================')
    print('==============================================================================')
    print('')

<class 'sklearn.linear_model._base.LinearRegression'> training begins

Fold 0: train=373 rows (50 to 422), test=368 rows (423 to 790), RMSE=2.4939, r2score_test=0.9852, r2score_train=0.9991
Fold 1: train=741 rows (50 to 790), test=368 rows (791 to 1158), RMSE=2.2384, r2score_test=0.9975, r2score_train=0.9980
Fold 2: train=1109 rows (50 to 1158), test=368 rows (1159 to 1526), RMSE=1.7912, r2score_test=0.9972, r2score_train=0.9986
Fold 3: train=1477 rows (50 to 1526), test=368 rows (1527 to 1894), RMSE=9.6108, r2score_test=0.9966, r2score_train=0.9993
Fold 4: train=1845 rows (50 to 1894), test=368 rows (1895 to 2262), RMSE=16.0266, r2score_test=0.9991, r2score_train=0.9997

Mean RMSE across folds: 6.4322

Mean R2_score_test across folds: 0.9952

Mean R2_score_train across folds: 0.9989
baseline information:  [np.float64(21.58164683631077), np.float64(72.81614746327485), np.float64(115.87767170359697), np.float64(577.2221524543197), np.float64(1656.4373988954264)] 488.7870034705858

<clas

We have to be careful with how we have trained using sector_index. Some of these data are cumulative increment which means the model will study general movement rather than actual trend. The good performance of the Non-tree based models could be due to leakages. We will have to revisit that.

In [29]:
## Lets deal with the lagging with respect to sector_index
fin_sec_data2=pd.read_csv('feature_engineering/stocks_sectors/financial_services/financial_services_sector')